# Sci-Bridge ON 사진 튜터 가능성 시험 (Kaggle)

학생이 찍은 실험 장면을 튜터가 보고 질문할 수 있을지 **학습 없이** 먼저 알아보는 시험입니다. 모델은 네이버 **HyperCLOVAX-SEED-Vision-Instruct-3B**(사진·영상 이해, 한국어)입니다.

**알아보는 것**
1. **기구 알아보기:** 사진 속 실험 기구와 물체를 제대로 말하는가
2. **예/아니오 확인:** "사진에 자석이 있나요?" 같은 물음에 맞게 답하는가. 없는 것을 있다고 하는 버릇(예 쏠림)도 셉니다
3. **튜터 질문:** 앱 튜터 규칙(1~2문장, 질문으로 끝냄, 정답·판정 없음)을 지키며 사진에서 볼 곳을 짚는가
4. **속도·크기:** 사진 한 장의 처리 시간, 입력 토큰 수, GPU 메모리

**개인정보 규칙 (꼭 지킵니다)**
- **학생이 찍은 사진, 학생 얼굴·이름표·교실이 나온 사진은 올리지 않습니다.** 이 시험은 공개 사진과 교사·연구자가 사람 없이 찍은 사진만 씁니다.
- 직접 찍은 사진은 Kaggle **비공개(Private)** 데이터셋으로 올립니다.
- 앱에 사진 기능을 넣게 되면 심박 데이터와 같은 조건을 따릅니다: 학생 허가, 기기 안에서만 처리(서버 전송 없음), 앱 종료 시 자동 삭제.

**준비**
1. Accelerator **GPU T4 x2**(한 장만 씁니다), Internet **On**.
2. (선택) 직접 찍은 실험 사진을 쓰려면 사진과 `images.json`을 한 폴더에 넣어 비공개 데이터셋으로 올리고 노트북에 추가합니다. 형식은 아래 설정 칸에 있습니다.
3. **Save & Run All**. 예시 사진만 쓰면 20~30분쯤 걸립니다(추정).

**라이선스:** 모델은 HyperCLOVA X SEED 라이선스, 예시 사진은 사진마다 적힌 위키미디어 공용 라이선스를 따릅니다(결과 페이지에 출처를 함께 적습니다).

In [ ]:
# ---- 설정 ----
import os, glob, json
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # torch를 불러오기 전에 GPU를 한 장으로 정한다
MODEL = "naver-hyperclovax/HyperCLOVAX-SEED-Vision-Instruct-3B"
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./vision-out"
USE_DEMO = True        # 위키미디어 공용 예시 사진을 쓴다
MY_IMAGES_DIR = None   # None이면 /kaggle/input 아래에서 images.json을 찾는다(없으면 예시 사진만)
MAX_NEW = 128          # 답 최대 토큰 수
# 직접 찍은 사진의 images.json 형식(사진 파일과 같은 폴더):
# [{"file": "magnet1.jpg", "unit": "자석의 이용 (3학년)", "activity": "막대자석에 클립 붙이기",
#   "expect": [["자석"], ["클립", "핀"]],                       # 설명에 나와야 할 낱말(안쪽 목록은 같은 뜻)
#   "checks": [["사진에 클립이 있나요?", "예"], ["사진에 비커가 있나요?", "아니오"]],
#   "forbidden": ["N극", "S극"]}]                                # 튜터 질문에 나오면 안 되는 정답 구절(선택)
if MY_IMAGES_DIR is None:
    hits = glob.glob("/kaggle/input/**/images.json", recursive=True)
    MY_IMAGES_DIR = os.path.dirname(hits[0]) if hits else None
os.makedirs(f"{OUT_DIR}/images", exist_ok=True)
print("OUT_DIR =", OUT_DIR, "| 내 사진 폴더 =", MY_IMAGES_DIR, "| 예시 사진 =", USE_DEMO)

In [ ]:
# ---- 패키지: 이 모델의 코드는 transformers 4.52.4에서 확인되었다(모델 카드). 학습 노트북과 따로 돌린다. ----
!pip install -q "transformers==4.52.4" "timm==1.0.15" "einops==0.8.1"

In [ ]:
import torch, time
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())
assert torch.cuda.device_count() <= 1, "GPU가 두 장 보여요. Restart & Clear Outputs 후 맨 위 칸부터 다시 실행해 주세요."
# T4(계산 능력 7.5)는 bf16이 느려 fp16을 쓴다. Ampere 이상이면 원본 형식인 bf16.
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print("dtype", DTYPE)

In [ ]:
# ---- 예시 사진: 위키미디어 공용에서 사람 얼굴이 없는 실험 장면만 골랐다(2026-11 확인) ----
DEMO = [
 {
  "id": "magnet-filings",
  "title": "File:Bar-magnets-attracting-iron-filings max.jpg",
  "page": "https://commons.wikimedia.org/wiki/File:Bar-magnets-attracting-iron-filings_max.jpg",
  "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/b/bf/Bar-magnets-attracting-iron-filings_max.jpg/960px-Bar-magnets-attracting-iron-filings_max.jpg",
  "author": "MikeRun, Vera Wurmsdobler",
  "license": "CC BY-SA 4.0",
  "unit": "자석의 이용 (3학년)",
  "activity": "막대자석 두 개 주위의 철가루 모양 관찰",
  "expect": [
   [
    "자석"
   ],
   [
    "철가루",
    "쇳가루",
    "가루"
   ]
  ],
  "checks": [
   [
    "사진에 막대자석이 있나요?",
    "예"
   ],
   [
    "사진에 전구가 있나요?",
    "아니오"
   ]
  ],
  "forbidden": [
   "끌어당기고 있",
   "자기력선"
  ]
 },
 {
  "id": "circuit-parts",
  "title": "File:9v battery and led circuit components (wires, battery, LED, resistor).jpg",
  "page": "https://commons.wikimedia.org/wiki/File:9v_battery_and_led_circuit_components_%28wires%2C_battery%2C_LED%2C_resistor%29.jpg",
  "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/b/b4/9v_battery_and_led_circuit_components_%28wires%2C_battery%2C_LED%2C_resistor%29.jpg/960px-9v_battery_and_led_circuit_components_%28wires%2C_battery%2C_LED%2C_resistor%29.jpg",
  "author": "D.M.C. Janiththa (phobia.tech)",
  "license": "CC BY-SA 4.0",
  "unit": "전기의 이용 (6학년)",
  "activity": "전기 회로 부품 살펴보기",
  "expect": [
   [
    "전지",
    "배터리"
   ],
   [
    "전선",
    "선"
   ],
   [
    "다이오드",
    "LED",
    "엘이디",
    "전구"
   ]
  ],
  "checks": [
   [
    "사진에 건전지가 있나요?",
    "예"
   ],
   [
    "사진에 온도계가 있나요?",
    "아니오"
   ]
  ],
  "forbidden": []
 },
 {
  "id": "lemon-battery",
  "title": "File:Lemon battery experiment.JPG",
  "page": "https://commons.wikimedia.org/wiki/File:Lemon_battery_experiment.JPG",
  "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/f/f9/Lemon_battery_experiment.JPG/960px-Lemon_battery_experiment.JPG",
  "author": "ChristianSW",
  "license": "CC BY 3.0",
  "unit": "전기의 이용 (6학년)",
  "activity": "레몬으로 발광 다이오드 불 켜기",
  "expect": [
   [
    "레몬"
   ],
   [
    "전선",
    "집게"
   ]
  ],
  "checks": [
   [
    "사진에 레몬이 있나요?",
    "예"
   ],
   [
    "사진에 건전지가 있나요?",
    "아니오"
   ]
  ],
  "forbidden": [
   "전해질",
   "전류가 흐르"
  ]
 },
 {
  "id": "cabbage-tubes",
  "title": "File:Indicateur chou rouge.jpg",
  "page": "https://commons.wikimedia.org/wiki/File:Indicateur_chou_rouge.jpg",
  "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/f/f7/Indicateur_chou_rouge.jpg/960px-Indicateur_chou_rouge.jpg",
  "author": "Supermartl; derivative work: Haltopub",
  "license": "CC BY-SA 3.0",
  "unit": "산과 염기 (6학년)",
  "activity": "붉은 양배추 지시약의 색깔 비교",
  "expect": [
   [
    "시험관",
    "유리관",
    "관"
   ],
   [
    "빨간",
    "빨강",
    "파란",
    "파랑",
    "초록",
    "노란",
    "노랑",
    "색"
   ]
  ],
  "checks": [
   [
    "사진에 시험관이 있나요?",
    "예"
   ],
   [
    "사진에 자석이 있나요?",
    "아니오"
   ]
  ],
  "forbidden": [
   "산성",
   "염기성"
  ]
 },
 {
  "id": "cabbage-lab",
  "title": "File:Extract of red cabbage in the beacker (pH indicator).jpg",
  "page": "https://commons.wikimedia.org/wiki/File:Extract_of_red_cabbage_in_the_beacker_%28pH_indicator%29.jpg",
  "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/e/ec/Extract_of_red_cabbage_in_the_beacker_%28pH_indicator%29.jpg/960px-Extract_of_red_cabbage_in_the_beacker_%28pH_indicator%29.jpg",
  "author": "Alessandro e Damiano",
  "license": "CC BY 4.0",
  "unit": "산과 염기 (6학년)",
  "activity": "붉은 양배추 지시약과 여러 용액",
  "expect": [
   [
    "비커"
   ],
   [
    "시험관"
   ],
   [
    "병",
    "플라스크"
   ]
  ],
  "checks": [
   [
    "사진에 비커가 있나요?",
    "예"
   ],
   [
    "사진에 촛불이 있나요?",
    "아니오"
   ]
  ],
  "forbidden": [
   "산성",
   "염기성"
  ]
 },
 {
  "id": "spring-scale",
  "title": "File:2010-10-19-Federwaage-1.JPG",
  "page": "https://commons.wikimedia.org/wiki/File:2010-10-19-Federwaage-1.JPG",
  "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/2/22/2010-10-19-Federwaage-1.JPG/960px-2010-10-19-Federwaage-1.JPG",
  "author": "R-E-AL",
  "license": "CC BY-SA 3.0",
  "unit": "물체의 무게 (4학년)",
  "activity": "용수철저울의 눈금 읽기",
  "expect": [
   [
    "저울"
   ],
   [
    "눈금",
    "바늘",
    "숫자"
   ]
  ],
  "checks": [
   [
    "사진에 갈고리가 있나요?",
    "예"
   ],
   [
    "사진에 비커가 있나요?",
    "아니오"
   ]
  ],
  "forbidden": []
 },
 {
  "id": "thermometer-cup",
  "title": "File:Measurement of water temperature.jpg",
  "page": "https://commons.wikimedia.org/wiki/File:Measurement_of_water_temperature.jpg",
  "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/e/e1/Measurement_of_water_temperature.jpg/960px-Measurement_of_water_temperature.jpg",
  "author": "Chong Tian Lin",
  "license": "CC BY-SA 4.0",
  "unit": "온도와 열 (5학년)",
  "activity": "컵에 든 물의 온도 재기",
  "expect": [
   [
    "온도계"
   ],
   [
    "컵",
    "물"
   ]
  ],
  "checks": [
   [
    "온도계 끝이 물에 잠겨 있나요?",
    "예"
   ],
   [
    "사진에 알코올 램프가 있나요?",
    "아니오"
   ]
  ],
  "forbidden": [
   "18"
  ]
 },
 {
  "id": "bean-sprout",
  "title": "File:Navy Bean Seed to Seedling Germination.jpg",
  "page": "https://commons.wikimedia.org/wiki/File:Navy_Bean_Seed_to_Seedling_Germination.jpg",
  "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/1/13/Navy_Bean_Seed_to_Seedling_Germination.jpg/960px-Navy_Bean_Seed_to_Seedling_Germination.jpg",
  "author": "B. Domangue",
  "license": "CC BY-SA 4.0",
  "unit": "식물의 한살이 (3학년)",
  "activity": "강낭콩이 싹 터서 자라는 모습 관찰",
  "expect": [
   [
    "콩",
    "씨"
   ],
   [
    "뿌리"
   ],
   [
    "잎",
    "싹"
   ]
  ],
  "checks": [
   [
    "사진에 뿌리가 보이나요?",
    "예"
   ],
   [
    "사진에 꽃이 있나요?",
    "아니오"
   ]
  ],
  "forbidden": []
 },
 {
  "id": "candle-glass",
  "title": "File:Candle Burning.JPG",
  "page": "https://commons.wikimedia.org/wiki/File:Candle_Burning.JPG",
  "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/1/19/Candle_Burning.JPG/960px-Candle_Burning.JPG",
  "author": "Nanda93",
  "license": "Public domain",
  "unit": "연소와 소화 (6학년)",
  "activity": "유리 그릇 속에서 타는 초 관찰",
  "expect": [
   [
    "초",
    "촛불",
    "양초"
   ],
   [
    "불"
   ]
  ],
  "checks": [
   [
    "사진에 불꽃이 있나요?",
    "예"
   ],
   [
    "사진에 시험관이 있나요?",
    "아니오"
   ]
  ],
  "forbidden": [
   "산소"
  ]
 },
 {
  "id": "rock-two",
  "title": "File:Granite-Basalt contact.jpg",
  "page": "https://commons.wikimedia.org/wiki/File:Granite-Basalt_contact.jpg",
  "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/f/fc/Granite-Basalt_contact.jpg/960px-Granite-Basalt_contact.jpg",
  "author": "Eurico Zimbres",
  "license": "CC BY-SA 2.5",
  "unit": "지층과 암석 (4학년)",
  "activity": "두 가지 암석의 알갱이와 색 비교",
  "expect": [
   [
    "돌",
    "암석",
    "바위"
   ],
   [
    "검은",
    "흰",
    "회색",
    "색"
   ]
  ],
  "checks": [
   [
    "사진에 돌이 있나요?",
    "예"
   ],
   [
    "사진에 물이 담긴 컵이 있나요?",
    "아니오"
   ]
  ],
  "forbidden": [
   "화강암",
   "현무암"
  ]
 }
]

import urllib.request
UA = {"User-Agent": "SciBridgeResearch/0.1 (educational prototype)"}   # 위키미디어 요청 규칙: 알아볼 수 있는 이름을 보낸다

def download(url, path):
    if os.path.exists(path): return path
    for attempt in range(5):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=60) as r, open(path, "wb") as f:
                f.write(r.read())
            return path
        except Exception as e:
            print("  다시 받기", attempt + 1, type(e).__name__); time.sleep(10 * (attempt + 1))
    raise RuntimeError(f"사진을 받지 못했어요: {url}")

ITEMS = []
if USE_DEMO:
    for d in DEMO:
        path = download(d["url"], f"{OUT_DIR}/images/{d['id']}.jpg"); time.sleep(1)
        ITEMS.append(dict(d, path=path, source="demo"))
if MY_IMAGES_DIR:
    for d in json.load(open(os.path.join(MY_IMAGES_DIR, "images.json"), encoding="utf-8")):
        ITEMS.append(dict(d, id=os.path.splitext(d["file"])[0], path=os.path.join(MY_IMAGES_DIR, d["file"]), source="mine"))
assert ITEMS, "시험할 사진이 없어요. USE_DEMO를 True로 두거나 images.json을 넣어 주세요."
print("사진", len(ITEMS), "장:", [x["id"] for x in ITEMS])

In [ ]:
# ---- 튜터 규칙 검사 (앱 packages/tutor의 check.ts·guardOutput과 같은 기준) ----
import re, unicodedata
MAX_CHARS, MAX_SENTENCES = 120, 2
JUDGMENT_WORDS = ["정답", "틀렸", "맞았", "잘했어", "오답"]
FOREIGN = re.compile(r"[\u3040-\u30ff\u3400-\u4dbf\u4e00-\u9fff\uf900-\ufaff]")   # 일본 글자·한자(앱 hasForeignScript와 같은 범위)
_norm = lambda s: "".join(ch for ch in unicodedata.normalize("NFC", s) if not (ch.isspace() or unicodedata.category(ch)[0] in "PSC"))
split_sentences = lambda t: [s.strip() for s in re.split(r"(?<=[.?!])\s+", t.strip()) if s.strip()]

def check_tutor(text, forbidden=()):
    t = text.strip(); p = []
    if not t or len(t) > MAX_CHARS or len(split_sentences(t)) > MAX_SENTENCES: p.append("too-long")
    if not any(s.endswith("?") for s in split_sentences(t)): p.append("no-question")
    if any(w in t for w in JUDGMENT_WORDS): p.append("judgment")
    if FOREIGN.search(t): p.append("foreign-script")
    p += [f"answer:{f}" for f in forbidden if _norm(f) and _norm(f) in _norm(t)]
    return p

def yes_no(text):
    t = text.strip()
    if t.startswith("아니") or "아니오" in t[:8] or "아니요" in t[:8] or "없" in t[:12]: return "아니오"
    if t.startswith("예") or t.startswith("네") or "있" in t[:12]: return "예"
    return "?"

def keyword_hits(text, expect):
    n = _norm(text)
    return [any(_norm(w) in n for w in group) for group in expect]

assert check_tutor("자석 주위의 철가루는 어떤 모양으로 늘어서 있나요?") == []
assert "no-question" in check_tutor("자석이에요.") and "answer:N극" in check_tutor("N극이 어디일까요?", ["N극"])
assert yes_no("예, 있어요.") == "예" and yes_no("아니오.") == "아니오"
assert not FOREIGN.search("가나다힣") and FOREIGN.search("变") and FOREIGN.search("の")

In [ ]:
# ---- 모델 불러오기 ----
from transformers import AutoModelForCausalLM, AutoProcessor, AutoTokenizer
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL, trust_remote_code=True, torch_dtype=DTYPE).to("cuda").eval()
processor = AutoProcessor.from_pretrained(MODEL, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
STOP = [tokenizer.convert_tokens_to_ids(t) for t in ["<|im_end|>", "<|endofturn|>", "<|stop|>"]]
print(f"불러오기 {time.time() - t0:.0f}초 | 가중치 {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B | GPU {torch.cuda.memory_allocated() / 2**30:.1f} GiB")

TUTOR_SYSTEM = (
    "너는 초등학교 과학 실험을 돕는 탐구 안내 선생님이다. 학생이 찍은 실험 사진과 말을 보고, 학생이 스스로 관찰하도록 돕는 탐침 질문만 한다.\n규칙:\n"
    "1. 한국어로 1~2문장만 쓰고, 반드시 질문으로 끝낸다. 120자를 넘기지 않는다.\n"
    "2. 정답, 결론, 보고서 문장을 말하지 않는다. 맞았는지 틀렸는지 판정하지 않는다.\n"
    "3. 학생이 배우지 않은 어려운 용어를 쓰지 않는다.\n"
    "4. 사진에서 학생이 살펴볼 곳(기구, 색, 모양, 눈금, 위치)을 구체적으로 짚어서 묻는다.\n"
    "5. 사진에 사람이 있어도 얼굴, 옷, 이름표는 말하지 않는다.")

@torch.no_grad()
def ask(item, text, system=None):
    chat = []
    if system: chat.append({"role": "system", "content": [{"type": "text", "text": system}]})
    chat.append({"role": "user", "content": [{"type": "image", "image": item["path"], "filename": os.path.basename(item["path"])}]})
    chat.append({"role": "user", "content": [{"type": "text", "text": text}]})
    inputs = processor.apply_chat_template(chat, tokenize=True, return_dict=True, return_tensors="pt", add_generation_prompt=True).to("cuda")
    torch.cuda.synchronize(); t = time.time()
    # 이 모델의 generate는 max_length를 '새로 만들 토큰 수'로 쓰고, 새로 만든 토큰만 돌려준다.
    out = model.generate(**inputs, max_length=MAX_NEW, do_sample=False, eos_token_id=STOP, pad_token_id=tokenizer.pad_token_id)
    torch.cuda.synchronize()
    ans = processor.batch_decode(out, skip_special_tokens=True)[0].strip()
    return ans, time.time() - t, int(inputs["input_ids"].shape[1])

In [ ]:
# ---- 사진마다 세 가지 시험 ----
DESCRIBE = "이 사진은 초등학교 과학 실험 장면이에요. 사진에 보이는 실험 기구와 물체를 두 문장 이내로 설명해 주세요."
rows = []
torch.cuda.reset_peak_memory_stats()
for it in ITEMS:
    r = dict(id=it["id"], source=it["source"], unit=it.get("unit"), activity=it.get("activity"))
    r["describe"], r["describe_sec"], r["input_tokens"] = ask(it, DESCRIBE)
    r["expect_hits"] = keyword_hits(r["describe"], it.get("expect", []))
    r["checks"] = []
    for q, want in it.get("checks", []):
        a, _, _ = ask(it, f"{q} '예' 또는 '아니오'로만 답하세요.")
        r["checks"].append(dict(q=q, want=want, answer=a, got=yes_no(a)))
    prompt = f"[단원] {it.get('unit', '')}\n[활동] {it.get('activity', '')}\n[학생] 실험 장면을 찍었어요. 뭘 봐야 해요?"
    r["tutor"], r["tutor_sec"], _ = ask(it, prompt, system=TUTOR_SYSTEM)
    r["tutor_problems"] = check_tutor(r["tutor"], it.get("forbidden", []))
    rows.append(r)
    print(f"\n[{it['id']}] {it.get('activity', '')}")
    print("  설명:", r["describe"], f"({r['describe_sec']:.1f}초, 입력 {r['input_tokens']}토큰)")
    for c in r["checks"]: print(f"  확인: {c['q']} → {c['answer']} (기대 {c['want']})")
    print("  튜터:", r["tutor"], r["tutor_problems"] or "규칙 통과")
with open(f"{OUT_DIR}/vision-results.jsonl", "w", encoding="utf-8") as f:
    for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [ ]:
# ---- 요약 ----
C = [c for r in rows for c in r["checks"]]
yes_items = [c for c in C if c["want"] == "예"]; no_items = [c for c in C if c["want"] == "아니오"]
hits = [h for r in rows for h in r["expect_hits"]]
summary = dict(
    images=len(rows),
    keyword_recall=sum(hits) / len(hits) if hits else None,                                  # 설명에 기대 낱말이 나온 비율
    yes_no_accuracy=sum(c["got"] == c["want"] for c in C) / len(C) if C else None,
    false_yes=sum(c["got"] == "예" for c in no_items) / len(no_items) if no_items else None,   # 없는 것을 있다고 한 비율
    missed_yes=sum(c["got"] != "예" for c in yes_items) / len(yes_items) if yes_items else None,
    tutor_rule_pass=sum(not r["tutor_problems"] for r in rows) / len(rows),
    tutor_answer_leak=sum(any(p.startswith("answer:") for p in r["tutor_problems"]) for r in rows),
    sec_per_answer=sum(r["describe_sec"] + r["tutor_sec"] for r in rows) / (2 * len(rows)),
    input_tokens=sum(r["input_tokens"] for r in rows) / len(rows),
    gpu_peak_gib=round(torch.cuda.max_memory_allocated() / 2**30, 2),
    gpu=torch.cuda.get_device_name(0), dtype=str(DTYPE))
json.dump(summary, open(f"{OUT_DIR}/vision-summary.json", "w"), ensure_ascii=False, indent=1)
PCT = {"keyword_recall", "yes_no_accuracy", "false_yes", "missed_yes", "tutor_rule_pass"}
fmt = lambda k, v: "-" if v is None else f"{v:.1%}" if k in PCT else f"{v:.1f}" if isinstance(v, float) else str(v)
labels = dict(images="사진 수", keyword_recall="기구 낱말 맞힘", yes_no_accuracy="예/아니오 정확도", false_yes="없는 것을 '예'",
              missed_yes="있는 것을 놓침", tutor_rule_pass="튜터 규칙 통과", tutor_answer_leak="튜터 정답 누출", sec_per_answer="답 하나당 초(T4)",
              input_tokens="사진 1장 입력 토큰", gpu_peak_gib="GPU 최대 메모리(GiB)", gpu="GPU", dtype="형식")
for k, v in summary.items(): print(f"{labels[k]:>16}: {fmt(k, v)}")

In [ ]:
# ---- 교사 검토용 페이지(vision-review.html): 사진, 모델 답, 출처를 한 장에 모은다 ----
import base64, html, io
from PIL import Image
def thumb(path):
    im = Image.open(path).convert("RGB"); im.thumbnail((480, 480)); b = io.BytesIO(); im.save(b, "JPEG", quality=80)
    return base64.b64encode(b.getvalue()).decode()
by_id = {x["id"]: x for x in ITEMS}
parts = ["<!doctype html><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'><title>사진 튜터 시험 결과</title>",
         "<style>body{font-family:sans-serif;max-width:900px;margin:auto;padding:16px;line-height:1.6}figure{border:1px solid #ccc;border-radius:8px;padding:12px;margin:16px 0}"
         "img{max-width:100%;height:auto}.bad{color:#b00}.ok{color:#070}small{color:#666}</style><h1>사진 튜터 시험 결과</h1>",
         "<p>" + html.escape(json.dumps(summary, ensure_ascii=False)) + "</p>"]
for r in rows:
    it = by_id[r["id"]]
    checks = "".join(f"<li>{html.escape(c['q'])} → {html.escape(c['answer'])} <span class='{'ok' if c['got'] == c['want'] else 'bad'}'>(기대 {c['want']})</span></li>" for c in r["checks"])
    credit = f"<small>출처: <a href='{html.escape(it['page'])}'>{html.escape(it['title'])}</a> · {html.escape(it['author'])} · {html.escape(it['license'])}</small>" if it.get("page") else "<small>직접 찍은 사진</small>"
    parts.append(f"<figure><img src='data:image/jpeg;base64,{thumb(it['path'])}' alt='{html.escape(r['activity'] or r['id'])}'><figcaption><b>{html.escape(r['activity'] or r['id'])}</b><br>"
                 f"설명: {html.escape(r['describe'])}<ul>{checks}</ul>튜터: {html.escape(r['tutor'])} "
                 f"<span class='{'bad' if r['tutor_problems'] else 'ok'}'>{html.escape(', '.join(r['tutor_problems']) or '규칙 통과')}</span><br>{credit}</figcaption></figure>")
open(f"{OUT_DIR}/vision-review.html", "w", encoding="utf-8").write("\n".join(parts))
print("저장:", f"{OUT_DIR}/vision-review.html", "| 결과:", f"{OUT_DIR}/vision-results.jsonl", f"{OUT_DIR}/vision-summary.json")

## 결과 읽는 법

| 지표 | 다음 단계로 갈 만한 수준(잠정) | 뜻 |
|---|---|---|
| 기구 낱말 맞힘 | 80% 이상 | 사진 속 실험 기구를 알아본다 |
| 예/아니오 정확도 | 85% 이상, **없는 것을 '예' 10% 이하** | 장치 점검 질문에 쓸 수 있다. '예' 쏠림이 크면 점검 기능은 쓰지 않는다 |
| 튜터 규칙 통과 | 70% 이상(학습 전) | 학습시키면 더 오를 여지가 있다. 정답 누출은 0이어야 한다 |
| 답 하나당 초 | 참고 | T4 기준. 태블릿·PC 속도는 따로 잰다 |

이 수준을 넘으면 사진 튜터 기능(사진 + 좋은 질문 견본, 기기 안 처리)을 설계합니다. 결과 파일 `vision-summary.json`, `vision-results.jsonl`, `vision-review.html`을 보내 주세요.